In [1]:
import os
import sys

PROJECT_ROOT = os.path.abspath("..")
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from sentence_transformers import SentenceTransformer

from utils import (
    build_binary_relevance_maps,
    build_eval_items,
    build_global_corpus,
    evaluate_sentence_transformer,
    evaluate_tfidf,
    load_triplets,
)

/upb/users/g/ganeshk/profiles/unix/cs/.conda/envs/sparql/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
DATA_PATH = os.path.join(PROJECT_ROOT, "dataset", "hard_triplets_tfidf_wikidata.json")
TOPKS = [3, 5, 10, 20]
BATCH = 32

MPNET_MODEL_NAME = "sentence-transformers/all-mpnet-base-v2"

raw_data = load_triplets(DATA_PATH)
global_corpus, seen_q = build_global_corpus(raw_data)
eval_items = build_eval_items(raw_data, global_corpus, seen_q)
relevance_scores_per_item = build_binary_relevance_maps(eval_items, global_corpus)

print(f"Loaded {len(raw_data)} items")
print(f"Global corpus size: {len(global_corpus)}")
print(f"Eval anchors with non-empty positives: {len(eval_items)}")

Loaded 186 items
Global corpus size: 2229
Eval anchors with non-empty positives: 186


In [3]:
evaluate_sentence_transformer(
    MPNET_MODEL_NAME,
    "Baseline: MPNet",
    eval_items,
    global_corpus,
    relevance_scores_per_item,
    TOPKS,
    batch_size=BATCH,
)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 4765.90it/s]
MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Baseline: MPNet
Binary relevance (annotated positives):
            P       R      F1     MRR    Hits   meanRel    nDCG
--------------------------------------------------------------
@3     0.0914  0.0172  0.0280  0.0986  0.1237    0.0914  0.0889
@5     0.0968  0.0326  0.0465  0.1021  0.1398    0.0968  0.0931
@10    0.0866  0.0575  0.0654  0.1109  0.2097    0.0866  0.0911
@20    0.0548  0.0722  0.0589  0.1141  0.2581    0.0548  0.0771
